In [8]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import lfilter

## 1 · Configuration

In [ ]:
# ── paths ─────────────────────────────────────────────────────────────────────
OPENSIM_ROOT = Path("/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/opensim-processing/data/AB01_Jinwoo")
NPZ_ROOT     = Path("/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/MeMo-IK-ID")

# ── subject parameters ────────────────────────────────────────────────────────
SUBJECT_MASS_KG = 88.0      # un-normalise model_out_nmpkg (N·m/kg) → N·m

# ── exoskeleton applied torque (subtract from OpenSim ID before model compare) ─
#   τ_exo [N·m] = model_out_nmpkg × SUBJECT_MASS_KG × scale
#   (same as (model × mass) × scale = model_nm × scale on the aligned grid).
# Keys must match the `exo` subfolder string in each TRIALS entry.
EXO_TORQUE_SCALE = {
    "hip-exo":  0.12,
    # "knee-exo": 0.09,
    "knee-exo": 0.12,
}

# ── mocap analog sample rate (Hz) ─────────────────────────────────────────────
MOCAP_FS = 1000.0

# ── ID low-pass filter — must match the controller's inference-path output LPF ──
# Both hip-exo-ctrl-V2/cfg/final.yaml and knee-exo-ctrl/cfg/final.yaml set:
#   out_lpf_hz: 6.0,  out_lpf_order: 4
# The controller uses _CausalLowPass: cascaded `out_lpf_order` × 1st-order EMA
# (RC-filter), applied sample-by-sample.  alpha = dt / (tau + dt),
# tau = 1 / (2π × cutoff_hz).  This is a *causal* filter — it introduces group
# delay and is NOT equivalent to a zero-phase Butterworth.  We replicate it
# exactly here so both signals have the same phase/delay before comparison.
ID_LPF_CUTOFF_HZ = 6.0     # must equal out_lpf_hz in the YAML
ID_LPF_ORDER     = 4       # must equal out_lpf_order in the YAML

# ── colour palette ─────────────────────────────────────────────────────────────
PALETTE = {
    "id":       "#2196F3",
    "model":    "#F44336",
    "residual": "#9C27B0",
    "gpio_m":   "#90A4AE",
    "gpio_n":   "#FF9800",
}

# ── trial mapping ──────────────────────────────────────────────────────────────
# Tuple: (exo_subfolder, opensim_cond_stem, id_moment_column, id_sign, delay_s)
#   exo_subfolder selects EXO_TORQUE_SCALE[exo] for τ_exo = nmpkg×mass×scale.
#
# id_sign : multiply raw OpenSim ID moment before LPF.
#   Hip  : +1  (same sign convention)
#   Knee : -1  (OpenSim extension-positive; controller flexion-positive)
#
# delay_s : exo output delay from hip-exo-ctrl-V2/cfg/final.yaml → `delay: 0.12`
#   model_out_nmpkg is recorded *before* the FIFO delay in the controller
#   (cascade.py line 317/347).  The delay is applied between model_out and the
#   motor command:  model_out → ×mass×scale → FIFO(delay_s) → torque LPF → applied
#   To compare model_out[t] against what the ID sees, we must sample the ID at
#   t + delay_s (i.e. look ahead by delay_s in the mocap/OpenSim time frame).
#   Knee YAML sets delay: 0 so no adjustment is needed there.
TRIALS = {
    "ab01_jinwoo_hip_0p8mps_lg_exo_on":  ("hip-exo",  "LG_0p8mps", "hip_flexion_r_moment", +1, 0.12),
    "ab01_jinwoo_hip_0p8mps_ra_exo_on":  ("hip-exo",  "RA_0p8mps", "hip_flexion_r_moment", +1, 0.12),
    "ab01_jinwoo_hip_1p2mps_lg_exo_on":  ("hip-exo",  "LG_1p2mps", "hip_flexion_r_moment", +1, 0.12),
    "ab01_jinwoo_hip_1p6mps_lg_exo_on":  ("hip-exo",  "LG_1p6mps", "hip_flexion_r_moment", +1, 0.12),
    "ab01_jinwoo_knee_0p8mps_ra_exo_on": ("knee-exo", "RA_0p8mps", "knee_angle_r_moment",  -1, 0.00),
    "ab01_jinwoo_knee_0p8mps_rd_exo_on": ("knee-exo", "RD_0p8mps", "knee_angle_r_moment",  -1, 0.00),
}

print("Trial manifest:")
for stem, (exo, cond, col, sign, delay) in TRIALS.items():
    files_ok = all([
        (NPZ_ROOT / f"{stem}.npz").exists(),
        (OPENSIM_ROOT / exo / "id"    / f"{cond}_id.sto").exists(),
        (OPENSIM_ROOT / exo / "mocap" / f"{cond}.csv").exists(),
    ])
    print(f"  {'✓' if files_ok else '✗'}  id_sign={sign:+d}  delay={delay:.2f}s  {stem}")

Trial manifest:
  ✓  id_sign=+1  delay=0.12s  ab01_jinwoo_hip_0p8mps_lg_exo_on
  ✓  id_sign=+1  delay=0.12s  ab01_jinwoo_hip_0p8mps_ra_exo_on
  ✓  id_sign=+1  delay=0.12s  ab01_jinwoo_hip_1p2mps_lg_exo_on
  ✓  id_sign=+1  delay=0.12s  ab01_jinwoo_hip_1p6mps_lg_exo_on
  ✓  id_sign=-1  delay=0.00s  ab01_jinwoo_knee_0p8mps_ra_exo_on
  ✓  id_sign=-1  delay=0.00s  ab01_jinwoo_knee_0p8mps_rd_exo_on


## 2 · Helper functions

In [10]:
def parse_opensim_sto(path: Path) -> pd.DataFrame:
    """Read an OpenSim .sto file; returns a DataFrame indexed by time."""
    with open(path) as fh:
        for i, line in enumerate(fh):
            if line.strip() == "endheader":
                header_end = i
                break
    df = pd.read_csv(path, sep=r"\s+", skiprows=header_end + 1)
    return df.set_index("time")


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS) -> tuple:
    """
    Parse a Vicon analog CSV (5-row header) and return
    (time_array [s], gpio_voltage_array [V]).
    """
    df = pd.read_csv(
        path, skiprows=[0, 1, 2, 4], header=0,
        low_memory=False, on_bad_lines="skip",
    )
    df = df[pd.to_numeric(df["Frame"], errors="coerce").notna()].copy()
    df["jet"] = pd.to_numeric(df["jet"], errors="coerce")
    df = df.dropna(subset=["jet"]).reset_index(drop=True)
    time = np.arange(len(df)) / fs
    return time, df["jet"].to_numpy(dtype=float)


def lpf_id(
    signal: np.ndarray,
    fs: float,
    cutoff_hz: float = ID_LPF_CUTOFF_HZ,
    order: int = ID_LPF_ORDER,
) -> np.ndarray:
    """
    Replicate the controller's _CausalLowPass exactly:
      cascaded `order` × 1st-order EMA (RC-filter), causal (no look-ahead).

      alpha = dt / (tau + dt),   tau = 1 / (2π × cutoff_hz)

    Each stage is implemented as scipy.signal.lfilter with
      b = [alpha],  a = [1, -(1-alpha)]
    and initial state set to signal[0] to match the controller's
    'initialize to first sample' behaviour.

    This matches hip-exo-ctrl-V2 and knee-exo-ctrl cascade.py _CausalLowPass,
    with out_lpf_hz=6.0 and out_lpf_order=4 from final.yaml.
    """
    dt    = 1.0 / float(fs)
    tau   = 1.0 / (2.0 * np.pi * float(cutoff_hz))
    alpha = dt / (tau + dt)

    b = np.array([alpha])
    a = np.array([1.0, -(1.0 - alpha)])

    y = signal.astype(float).copy()
    for _ in range(order):
        # initialise filter state so y[0] == signal[0] (matches controller init)
        zi = np.array([y[0] * (1.0 - alpha)])
        y, _ = lfilter(b, a, y, zi=zi)
    return y.astype(signal.dtype)


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> int | None:
    """Index of the first sample immediately after a falling edge."""
    above = signal > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


print("Helpers defined.")

Helpers defined.


## 3 · Pre-load & sync all trials

Runs once. Each trial is parsed, GPIO-synced, and ID-filtered. Interpolated ID has exo torque subtracted (`τ_exo` = `model_out_nmpkg` × mass × `EXO_TORQUE_SCALE[exo]`) before comparison; results are cached in `TRIAL_DATA`.

In [11]:
TRIAL_DATA = {}   # stem → dict with aligned arrays

for stem, (exo, cond, moment_col, id_sign, delay_s) in TRIALS.items():
    npz_path   = NPZ_ROOT / f"{stem}.npz"
    id_path    = OPENSIM_ROOT / exo / "id"    / f"{cond}_id.sto"
    mocap_path = OPENSIM_ROOT / exo / "mocap" / f"{cond}.csv"

    if not all(p.exists() for p in (npz_path, id_path, mocap_path)):
        print(f"[SKIP] {stem} — missing file(s)")
        continue

    print(f"Loading {stem} …", end=" ", flush=True)

    # NPZ
    npz         = np.load(npz_path)
    t_npz       = npz["time"]
    gpio_npz    = npz["GPIO"]
    model_nm    = npz["model_out_nmpkg"] * SUBJECT_MASS_KG   # N·m
    exo_scale   = EXO_TORQUE_SCALE[exo]

    # Mocap GPIO
    t_mocap, gpio_mocap = parse_mocap_csv(mocap_path)
    g_range = gpio_mocap.max() - gpio_mocap.min()
    gpio_mocap_norm = (gpio_mocap - gpio_mocap.min()) / g_range if g_range > 0 else gpio_mocap

    # OpenSim ID — sign-correct, then LPF
    id_df         = parse_opensim_sto(id_path)
    t_id          = id_df.index.to_numpy(float)
    id_fs         = 1.0 / float(np.median(np.diff(t_id)))
    id_moment_raw = id_df[moment_col].to_numpy(float) * id_sign
    id_moment     = lpf_id(id_moment_raw, fs=id_fs)

    # GPIO sync: maps NPZ time → ID/mocap time frame
    idx_npz   = first_falling_edge(gpio_npz)
    idx_mocap = first_falling_edge(gpio_mocap_norm)
    if idx_npz is None or idx_mocap is None:
        print(f"[WARN] no GPIO falling edge — skipping")
        continue
    t_offset      = t_mocap[idx_mocap] - t_npz[idx_npz]
    t_npz_aligned = t_npz + t_offset   # NPZ time expressed in the ID time frame

    # Delay correction: model_out_nmpkg is recorded before the FIFO delay.
    # The motor actually applies model_out at t_npz_aligned + delay_s.
    # So we evaluate the ID at (t_npz_aligned + delay_s) for each model sample.
    t_id_eval = t_npz_aligned - delay_s

    # Overlap: keep only samples where the delayed ID lookup is within range
    mask        = (t_id_eval >= t_id[0]) & (t_id_eval <= t_id[-1])
    t_common    = t_npz_aligned[mask]    # x-axis: model prediction time (ID frame)
    model_common  = model_nm[mask]
    exo_torque_nm = model_common * exo_scale   # nmpkg × mass × scale

    # Interpolate ID at the delay-shifted times
    id_interp = interp1d(t_id, id_moment, kind="linear",
                         bounds_error=False, fill_value=np.nan)(t_id_eval[mask])
    id_nm_net = id_interp - exo_torque_nm     # ID minus applied exo torque

    TRIAL_DATA[stem] = {
        "stem":            stem,
        "moment_col":      moment_col,
        "id_sign":         id_sign,
        "delay_s":         delay_s,
        "exo_scale":       exo_scale,
        "t":               t_common,
        "t_rel":           t_common - t_common[0],
        "model_nm":        model_common,
        "id_nm_gross":     id_interp,
        "exo_torque_nm":   exo_torque_nm,
        "id_nm":           id_nm_net,
        "t_offset":        t_offset,
        "t_mocap_rel":     t_mocap - t_mocap[0],
        "gpio_mocap_norm": gpio_mocap_norm,
        "t_npz_rel":       t_npz - t_npz[0],
        "gpio_npz":        gpio_npz,
    }
    dur = float(t_common[-1] - t_common[0])
    print(f"ok  ({dur:.1f} s overlap, t_offset={t_offset:+.3f} s, delay={delay_s:.3f} s)")

print(f"\nLoaded {len(TRIAL_DATA)} / {len(TRIALS)} trials.")

Loading ab01_jinwoo_hip_0p8mps_lg_exo_on … ok  (75.0 s overlap, t_offset=-0.233 s, delay=0.120 s)
Loading ab01_jinwoo_hip_0p8mps_ra_exo_on … ok  (79.7 s overlap, t_offset=-0.146 s, delay=0.120 s)
Loading ab01_jinwoo_hip_1p2mps_lg_exo_on … ok  (75.0 s overlap, t_offset=-0.238 s, delay=0.120 s)
Loading ab01_jinwoo_hip_1p6mps_lg_exo_on … ok  (75.0 s overlap, t_offset=-0.236 s, delay=0.120 s)
Loading ab01_jinwoo_knee_0p8mps_ra_exo_on … ok  (75.0 s overlap, t_offset=-0.230 s, delay=0.000 s)
Loading ab01_jinwoo_knee_0p8mps_rd_exo_on … ok  (72.0 s overlap, t_offset=-1.707 s, delay=0.000 s)

Loaded 6 / 6 trials.


## 4 · Interactive comparison

In [12]:
# ── widget definitions ────────────────────────────────────────────────────────
_first_stem = next(iter(TRIAL_DATA))
_first_data = TRIAL_DATA[_first_stem]

trial_dropdown = widgets.Dropdown(
    options=list(TRIAL_DATA.keys()),
    value=_first_stem,
    description="Trial:",
    layout=widgets.Layout(width="620px"),
    style={"description_width": "80px"},
)

time_slider = widgets.FloatRangeSlider(
    value=[_first_data["t_rel"][0], _first_data["t_rel"][-1]],
    min=_first_data["t_rel"][0],
    max=_first_data["t_rel"][-1],
    step=0.5,
    description="Time (s):",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "80px"},
    continuous_update=False,
    readout_format=".1f",
)

show_gpio_toggle = widgets.ToggleButton(
    value=False,
    description="Show GPIO panel",
    button_style="",
    icon="signal",
    layout=widgets.Layout(width="160px"),
)

out = widgets.Output()


# ── draw function ─────────────────────────────────────────────────────────────
def _draw(stem: str, t0: float, t1: float, show_gpio: bool) -> None:
    d    = TRIAL_DATA[stem]
    mask = (d["t_rel"] >= t0) & (d["t_rel"] <= t1)
    t    = d["t_rel"][mask]
    model = d["model_nm"][mask]
    id_m  = d["id_nm"][mask]
    resid = model - id_m

    valid = ~np.isnan(id_m)
    rmse  = float(np.sqrt(np.nanmean((model[valid] - id_m[valid])**2))) if valid.any() else np.nan
    r2    = float(np.corrcoef(model[valid], id_m[valid])[0, 1])**2      if valid.sum() > 1 else np.nan
    peak_m = float(np.nanmax(np.abs(model[valid]))) if valid.any() else np.nan
    peak_i = float(np.nanmax(np.abs(id_m[valid])))  if valid.any() else np.nan

    n_rows = 3 if show_gpio else 2
    row_h  = [2.8, 1.6, 1.5] if show_gpio else [2.8, 1.6]
    fig, axs = plt.subplots(
        n_rows, 1, figsize=(15, sum(row_h)),
        sharex=False,
        gridspec_kw={"height_ratios": row_h, "hspace": 0.45},
    )

    lpf_tag = f"{ID_LPF_CUTOFF_HZ:.0f} Hz LPF"
    id_plot_label = (
        f"OpenSim ID − τ_exo — {d['moment_col']} (×{d['id_sign']:+d}, {lpf_tag}, "
        f"+{d['delay_s']*1000:.0f} ms delay; τ_exo = model×mass×{d['exo_scale']:.3f})"
    )

    # ── subplot 0: moment overlay ─────────────────────────────────────────────
    axs[0].plot(t, id_m,  color=PALETTE["id"],    lw=2.0, label=id_plot_label)
    axs[0].plot(t, model, color=PALETTE["model"],  lw=1.8, ls="--",
                label=f"Model × {SUBJECT_MASS_KG:.0f} kg")
    axs[0].set_ylabel("Moment (N·m)", fontsize=12)
    axs[0].set_title(
        f"{stem}\n"
        f"RMSE = {rmse:.2f} N·m   R² = {r2:.3f}   "
        f"peak model = {peak_m:.1f} N·m  |  peak ID−τ_exo ({lpf_tag}) = {peak_i:.1f} N·m  |  R² = {r2:.3f}",
        fontsize=10,
    )
    axs[0].legend(fontsize=10, loc="upper right")
    axs[0].axhline(0, color="gray", lw=0.7, ls=":")
    axs[0].spines[["top", "right"]].set_visible(False)
    axs[0].tick_params(labelsize=10)
    axs[0].set_xlabel("Time (s)", fontsize=11)

    # ── subplot 1: residual ───────────────────────────────────────────────────
    axs[1].plot(t, resid, color=PALETTE["residual"], lw=1.5,
                label="Model − (ID − τ_exo)")
    axs[1].axhline(0, color="black", lw=0.8, ls=":")
    axs[1].set_ylabel("Residual (N·m)", fontsize=12)
    axs[1].set_xlabel("Time (s)", fontsize=11)
    axs[1].set_title(f"Residual = Model − (ID − τ_exo)  ({lpf_tag})", fontsize=10)
    axs[1].legend(fontsize=10, loc="upper right")
    axs[1].spines[["top", "right"]].set_visible(False)
    axs[1].tick_params(labelsize=10)

    # ── subplot 2: GPIO diagnostic (optional) ─────────────────────────────────
    if show_gpio:
        axs[2].plot(d["t_mocap_rel"], d["gpio_mocap_norm"],
                    color=PALETTE["gpio_m"], lw=1.0, alpha=0.8,
                    label="Mocap GPIO (normalised)")
        axs[2].plot(d["t_npz_rel"], d["gpio_npz"],
                    color=PALETTE["gpio_n"], lw=1.3, ls="--",
                    label="NPZ GPIO")
        axs[2].set_ylabel("Amplitude (a.u.)", fontsize=11)
        axs[2].set_xlabel("Time from recording start (s)", fontsize=11)
        axs[2].set_title(
            f"GPIO sync pulse  (t_offset = {d['t_offset']:+.4f} s)", fontsize=10
        )
        axs[2].legend(fontsize=10, loc="upper right")
        axs[2].spines[["top", "right"]].set_visible(False)
        axs[2].tick_params(labelsize=10)

    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    out.clear_output(wait=True)
    with out:
        display(Image(data=buf.read()))


# ── callbacks ─────────────────────────────────────────────────────────────────
def _on_trial_change(change):
    stem = change["new"]
    d    = TRIAL_DATA[stem]
    t_rel = d["t_rel"]
    time_slider.min   = float(t_rel[0])
    time_slider.max   = float(t_rel[-1])
    time_slider.value = [float(t_rel[0]), float(t_rel[-1])]
    _redraw()

def _redraw(*_):
    _draw(trial_dropdown.value, *time_slider.value, show_gpio_toggle.value)

trial_dropdown.observe(_on_trial_change, names="value")
time_slider.observe(_redraw, names="value")
show_gpio_toggle.observe(_redraw, names="value")

# ── layout & initial render ───────────────────────────────────────────────────
controls = widgets.HBox(
    [trial_dropdown, show_gpio_toggle],
    layout=widgets.Layout(align_items="center", gap="12px"),
)
display(controls, time_slider, out)
_redraw()

FloatRangeSlider(value=(0.0, 74.98000467800011), continuous_update=False, description='Time (s):', layout=Layo…

Output()

/var/folders/hs/2_f3r_wj2mnd_3b0wcc41v2w0000gn/T/ipykernel_99709/272513150.py:107: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


## 5 · Window statistics

Re-run this cell after adjusting the slider to get stats for the current time window.

In [13]:
t0_s, t1_s = time_slider.value
stem_s     = trial_dropdown.value
d_s        = TRIAL_DATA[stem_s]

mask_s = (d_s["t_rel"] >= t0_s) & (d_s["t_rel"] <= t1_s)
model_s = d_s["model_nm"][mask_s]
id_s    = d_s["id_nm"][mask_s]
resid_s = model_s - id_s
valid_s = ~np.isnan(id_s)

lpf_tag = f"{ID_LPF_CUTOFF_HZ:.0f} Hz LPF"

signals_s = {
    f"OpenSim ID − τ_exo  ({d_s['moment_col']}, scale={d_s['exo_scale']}, {lpf_tag})  [N·m]": id_s[valid_s],
    f"Model output × {SUBJECT_MASS_KG:.0f} kg  [N·m]":      model_s[valid_s],
    f"Residual (Model − (ID − τ_exo))  [N·m]":               resid_s[valid_s],
}

rmse_s = float(np.sqrt(np.nanmean(resid_s[valid_s]**2)))
r2_s   = float(np.corrcoef(model_s[valid_s], id_s[valid_s])[0, 1])**2 if valid_s.sum() > 1 else np.nan

print(f"Trial  : {stem_s}")
print(f"Window : {t0_s:.1f} s – {t1_s:.1f} s  ({t1_s - t0_s:.1f} s)")
print(f"RMSE   : {rmse_s:.3f} N·m     R² = {r2_s:.4f}")
print()
print(f"{'Signal':<55} {'min':>8} {'max':>8} {'mean':>8} {'std':>8}")
print("-" * 91)
for name, sig in signals_s.items():
    print(f"{name:<55} {sig.min():8.2f} {sig.max():8.2f} {sig.mean():8.2f} {sig.std():8.2f}")

Trial  : ab01_jinwoo_hip_0p8mps_lg_exo_on
Window : 0.0 s – 75.0 s  (75.0 s)
RMSE   : 15.277 N·m     R² = 0.7031

Signal                                                       min      max     mean      std
-------------------------------------------------------------------------------------------
OpenSim ID  (hip_flexion_r_moment, 6 Hz LPF)  [N·m]       -45.25    23.59    -8.79    15.08
Model output × 88 kg  [N·m]                               -39.45    45.91     2.73    18.39
Residual (Model − ID)  [N·m]                              -20.28    45.44    11.53    10.03


## 6 · All-trials summary (full overlap window)

In [14]:
import pandas as pd

rows = []
for stem, d in TRIAL_DATA.items():
    valid = ~np.isnan(d["id_nm"])
    m, i  = d["model_nm"][valid], d["id_nm"][valid]
    rmse  = float(np.sqrt(np.mean((m - i)**2)))
    r2    = float(np.corrcoef(m, i)[0, 1])**2 if valid.sum() > 1 else np.nan
    rows.append({
        "trial":              stem,
        "moment_col":         d["moment_col"],
        "exo_scale":          d["exo_scale"],
        f"ID LPF":            f"{ID_LPF_CUTOFF_HZ:.0f} Hz {ID_LPF_ORDER}-ord",
        "RMSE [N·m]":        round(rmse, 2),
        "R²":                  round(r2, 3),
        "peak_model [N·m]":  round(float(np.nanmax(np.abs(m))), 1),
        "peak_ID_net [N·m]": round(float(np.nanmax(np.abs(i))), 1),
        "overlap [s]":        round(float(d["t_rel"][-1] - d["t_rel"][0]), 1),
        "t_offset [s]":       round(float(d["t_offset"]), 4),
    })

pd.DataFrame(rows)

,trial,moment_col,ID LPF,RMSE [N·m],R²,peak_model [N·m],peak_ID [N·m],overlap [s],t_offset [s]
0,ab01_jinwoo_hip_0p8mps_lg_exo_on,hip_flexion_r_moment,6 Hz 4-ord,15.28,0.703,45.9,45.2,75.0,-0.2331
1,ab01_jinwoo_hip_0p8mps_ra_exo_on,hip_flexion_r_moment,6 Hz 4-ord,19.39,0.742,72.1,77.4,79.7,-0.1461
2,ab01_jinwoo_hip_1p2mps_lg_exo_on,hip_flexion_r_moment,6 Hz 4-ord,11.97,0.924,57.1,55.8,75.0,-0.2380
3,ab01_jinwoo_hip_1p6mps_lg_exo_on,hip_flexion_r_moment,6 Hz 4-ord,7.65,0.975,72.5,80.9,75.0,-0.2360
4,ab01_jinwoo_knee_0p8mps_ra_exo_on,knee_angle_r_moment,6 Hz 4-ord,36.18,0.044,63.4,69.8,75.0,-0.2300
5,ab01_jinwoo_knee_0p8mps_rd_exo_on,knee_angle_r_moment,6 Hz 4-ord,33.34,0.321,82.9,60.3,72.0,-1.7070
